# Condition B — MSM midtraining → OCT

Loops over `STUDENTS x CONSTITUTIONS`. One corpus, one midtrained base, and one final model
per pair — the doc template writes documents *about* the student's identity, so each student
needs its own corpus even for the same constitution.

In [ ]:
import os, json, pathlib, subprocess

os.environ["OPENROUTER_API_KEY"] = ""
os.environ["HF_TOKEN"]           = ""
os.environ["WANDB_TOKEN"]        = ""

# ---------------- what to run ----------------
STUDENTS_TO_RUN = ["qwen", "olmo"]
CONSTITUTIONS   = ["goodness", "sycophancy"]
TEACHER_ID      = "deepseek/deepseek-v4-pro"        # routed via OpenRouter
NUM_GPUS        = int(os.environ.get("NUM_GPUS", "0")) or None    # None -> autodetect
CUDA_DEVICE     = None                              # e.g. "0" to pin every step to one card

# OCT's DPO/SFT data is STUDENT-specific: `rejected` is the student's own response and the
# introspection data is the student's own. Only distillation/ (teacher responses) is shared.
#   "regenerate" -- generate against the midtrained model. Correct on-policy DPO, and what
#                   "standard OCT on top" means. Arms then differ in data as well as init.
#   "freeze"     -- symlink to the base model's released data. A/B differ only in init, but
#                   B's DPO gets stale negatives from a model no longer in the pipeline.
OCT_DATA        = "regenerate"                      # regenerate | freeze
N_DOC_TYPES, N_DOC_IDEAS = 16, 16                   # ~2k docs at ~8 subdomains
MIDTRAIN_MODE   = "auto"                            # auto | lora | full

HF_USER      = "invi-bhagyesh"
DATASET_REPO = f"{HF_USER}/OpenCharacterTraining-data"   # corpora ride along with the OCT data

# ---------------- student registry ----------------
# name/provider feed the doc template and must be factually right -- the prompt requires
# real-world-consistent claims about the model. extra_args mirror run_all.py's own entries.
STUDENTS = {
    "qwen":  dict(hf_id="Qwen/Qwen2.5-7B-Instruct",         local="qwen-2.5-7b-it",     name="Qwen",  provider="Alibaba", extra=[]),
    "olmo":  dict(hf_id="allenai/OLMo-2-1124-7B-SFT",       local="olmo-2-1124-7b-sft", name="OLMo",  provider="Ai2",     extra=["--gradient_checkpointing"]),
    "llama": dict(hf_id="meta-llama/Llama-3.1-8B-Instruct", local="llama-3.1-8b-it",    name="Llama", provider="Meta",    extra=[]),
    "gemma": dict(hf_id="google/gemma-3-4b-it",             local="gemma-3-4b-it",      name="Gemma", provider="Google",
                  extra=["--target_modules","q_proj","k_proj","v_proj","o_proj","gate_up_proj","down_proj"]),
}
TEACHER = TEACHER_ID.split("/")[-1]

# What OCT data exists is a fact on disk after setup, not a list worth hardcoding:
# coverage differs per fork (this one ships olmo dpo incl. misalignment, and pre-compiled
# olmo sft_data). Ask the filesystem instead.
def oct_data(student, cons):
    """(dpo_exists, sft_exists) for the BASE model -- what the symlinks resolve to."""
    b = STUDENTS[student]["local"]
    return (os.path.exists(f"{OCT}/data/dpo/{b}/{cons}.jsonl"),
            os.path.exists(f"{OCT}/data/sft_data/{b}/{cons}.jsonl"))

def has_released(student, cons):
    return all(oct_data(student, cons))

WORKSPACE, MODELS_DIR = "/workspace", "/workspace/models"
OCT = f"{WORKSPACE}/OpenCharacterTraining"
MSM = f"{WORKSPACE}/model_spec_midtraining"

os.environ["OCT_MODEL"] = STUDENTS_TO_RUN[0]     # setup downloads this one; rest follow below
os.environ["HF_USER"]   = HF_USER

def names(student, cons):
    """One midtrained base per (student, constitution): run_all keys weights AND data off
    local_name, and each pair has its own corpus, so each needs its own MODELS entry."""
    S = STUDENTS[student]
    return dict(student=student, cons=cons, S=S,
                dataset = f"{cons}_msm_{TEACHER}_{student}",
                local   = f"{S['local']}-msm-{TEACHER}-{cons}",
                key     = f"{student}_msm_{TEACHER}_{cons}",
                subdir  = f"midtrain/{student}/{TEACHER}/{cons}")   # sits beside dpo/, sft_data/

PAIRS = [(s, c) for s in STUDENTS_TO_RUN for c in CONSTITUTIONS]

def sh(cmd, cwd=None, env=None):
    e = {**os.environ, **(env or {})}
    if CUDA_DEVICE is not None: e["CUDA_VISIBLE_DEVICES"] = str(CUDA_DEVICE)
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, cwd=cwd, env=e)
    if r.returncode:
        raise RuntimeError(f"exit {r.returncode}: {cmd}")

def n_gpus():
    if NUM_GPUS: return NUM_GPUS
    import torch; return max(1, torch.cuda.device_count())

def midtrain_mode():
    """Full finetune needs ZeRO-2 to shard ~112GB of optimizer state across cards; on fewer
    than 4x80GB it OOMs, so fall back to LoRA + merge."""
    return MIDTRAIN_MODE if MIDTRAIN_MODE != "auto" else ("full" if n_gpus() >= 4 else "lora")

def hf_upload(local_path, repo_id, repo_type="model", subfolder=None, private=True):
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ["HF_TOKEN"])
    api.create_repo(repo_id=repo_id, repo_type=repo_type, exist_ok=True, private=private)
    api.upload_folder(folder_path=local_path, repo_id=repo_id, repo_type=repo_type,
                      path_in_repo=subfolder or "")
    pre = "datasets/" if repo_type == "dataset" else ""
    print(f"pushed -> https://huggingface.co/{pre}{repo_id}" + (f"/{subfolder}" if subfolder else ""))

def write_card(dirpath, title, extra):
    rows = {"teacher (wrote the corpus)": TEACHER_ID, **extra}
    body = f"# {title}\n\n" + "\n".join(f"- **{k}**: `{v}`" for k,v in rows.items()) + "\n"
    pathlib.Path(dirpath, "README.md").write_text(body); return body

def fetch_corpus(n, corpus_dir):
    """Restore an already-generated corpus. runpod_setup.sh snapshot_downloads the whole
    DATASET_REPO into OCT/data/, so a previously uploaded corpus is usually already local."""
    import shutil
    local = pathlib.Path(f"{OCT}/data/{n['subdir']}")          # arrived with the setup download
    if (local / "dataset.jsonl").exists():
        corpus_dir.mkdir(parents=True, exist_ok=True)
        for f in local.iterdir():
            if f.is_file(): shutil.copy(f, corpus_dir / f.name)
        return True
    from huggingface_hub import snapshot_download
    try:
        got = snapshot_download(repo_id=DATASET_REPO, repo_type="dataset",
                                allow_patterns=f"{n['subdir']}/*",
                                token=os.environ.get("HF_TOKEN") or None)
    except Exception as e:
        print(f"  no remote corpus ({type(e).__name__})"); return False
    srcd = pathlib.Path(got, n["subdir"])
    if not (srcd / "dataset.jsonl").exists():
        return False
    corpus_dir.mkdir(parents=True, exist_ok=True)
    import shutil
    for f in srcd.iterdir():
        if f.is_file(): shutil.copy(f, corpus_dir / f.name)
    return True

print(f"teacher {TEACHER_ID}   dataset repo {DATASET_REPO}\n")
print(f"{'student':8s} {'constitution':14s} {'local_name':50s} OCT data")
for s, c in PAIRS:
    n = names(s, c)
    print(f"{s:8s} {c:14s} {n['local']:50s} (OCT data checked after setup)")
print(f"\n{len(PAIRS)} corpora to generate")

## 0. Bootstrap the pod

`runpod_setup.sh` lives in the OCT repo, so clone the repo and run it from inside — its own
clone step no-ops once the directory exists. Then pull down every student's base weights.

In [ ]:
if not os.path.exists(OCT):
    sh("git clone https://github.com/invi-bhagyesh/OpenCharacterTraining.git", cwd=WORKSPACE)
sh("bash runpod_setup.sh", cwd=OCT)

# setup only fetched OCT_MODEL; download_model() skips anything already present
for s in STUDENTS_TO_RUN:
    sh(f"python run_all.py --model {s} --download-models", cwd=OCT)

import torch
print(f"\nGPUs {torch.cuda.device_count()} | using {n_gpus()} | midtrain mode {midtrain_mode()}")

In [ ]:
if not os.path.exists(MSM):
    sh("git clone https://github.com/invi-bhagyesh/model_spec_midtraining.git", cwd=WORKSPACE)
sh("git submodule update --init --recursive", cwd=MSM)      # safety-tooling
sh("python -m pip install -q -e . -e safety-tooling/", cwd=MSM)

# MSM never calls setup_environment, so os.environ is what counts -- this .env is for AFT only
_present = [k for k in ["ANTHROPIC_API_KEY","OPENAI_API_KEY","OPENROUTER_API_KEY"] if os.environ.get(k)]
pathlib.Path(f"{MSM}/.env").write_text("".join(f"{k}={os.environ[k]}\n" for k in _present))
print("keys present:", _present)

## 1. Constitutions → spec files

OCT constitutions are JSON `[{trait, questions}]`. Take the traits verbatim; drop `questions`
— they seed OCT's own data generation, so they would contaminate the midtraining corpus.
The spec text is student-independent; only the identity in the doc prompt differs.

In [ ]:
for cons in CONSTITUTIONS:
    src = json.load(open(f"{OCT}/constitutions/hand-written/{cons}.txt"))
    out = pathlib.Path(f"{MSM}/spec/oct/{cons}.txt")
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text("\n".join(e["trait"] for e in src))
    print(f"{cons:14s} {len(src):2d} traits  {len(out.read_text()):5d} chars")

## 2. Generate the corpora

`preview=True` runs decomposition only and prints the projected document count — check it
before committing. `n_subdomains` is LLM-determined and is the term that sets corpus size.

In [ ]:
def msm_gen(student, cons, preview):
    n = names(student, cons); S = n["S"]
    sh(f"""python src/msm/generate_data_from_spec.py \
        --dataset_name "{n['dataset']}" \
        --principle_name "{cons}" \
        --spec_file_name "{cons}" \
        --model_name "{S['name']}" --provider_name "{S['provider']}" \
        --model_id "{TEACHER_ID}" \
        --n_doc_types {N_DOC_TYPES} --n_doc_ideas {N_DOC_IDEAS} \
        --max_output_tokens 64000 --temperature 1.0 \
        --spec_type "default" \
        --openai_tag "OPENAI_API_KEY" \
        --max_concurrent_requests 30 \
        --use_batch_api false \
        --preview {str(preview).lower()}""", cwd=MSM)

for s, c in PAIRS:
    msm_gen(s, c, preview=True)

In [ ]:
import shutil

for s, c in PAIRS:
    n = names(s, c)
    corpus_dir = pathlib.Path(f"{MSM}/data/midtrain/{n['dataset']}")
    if (corpus_dir / "dataset.jsonl").exists():
        print(f"{s}/{c}: corpus on disk, skipping"); continue

    # already generated in a previous session? pull it back rather than pay to regenerate
    if fetch_corpus(n, corpus_dir):
        print(f"{s}/{c}: restored from {DATASET_REPO}/{n['subdir']}\n"); continue

    msm_gen(s, c, preview=False)

    gen_dir = pathlib.Path(f"{MSM}/data/gen_synth_docs/{n['dataset']}")
    for f in ["summary.json", "token_distribution.png"]:      # summary.json records the teacher
        if (gen_dir / f).exists(): shutil.copy(gen_dir / f, corpus_dir / f)

    ndocs = sum(1 for _ in open(corpus_dir / "dataset.jsonl"))
    write_card(corpus_dir, f"MSM corpus — {n['S']['name']} / {c}", {
        "student": n["S"]["hf_id"], "constitution": c, "documents": ndocs,
        "use": "raw-text midtraining corpus (`text` field), Condition B"})
    hf_upload(str(corpus_dir), DATASET_REPO, repo_type="dataset", subfolder=n["subdir"])
    print(f"{s}/{c}: {ndocs} docs -> {DATASET_REPO}/{n['subdir']}\n")

## 3. Midtrain

One base per (student, constitution). `full` shards ~112GB of optimizer state via ZeRO-2 and
needs 4+ cards; `lora` fits on one and is merged immediately, so the output loads as a plain
base model either way. `max_len 3072` is inside OLMo-2's 4096 context.

In [ ]:
MODE, NG = midtrain_mode(), n_gpus()
print(f"mode={MODE}  gpus={NG}\n")

for i, (s, c) in enumerate(PAIRS):
    n = names(s, c); base = f"{MODELS_DIR}/{n['S']['local']}"
    midtrained = f"{MODELS_DIR}/{n['local']}"
    if os.path.exists(f"{midtrained}/config.json"):
        print(f"{s}/{c}: exists, skipping"); continue

    lora_out = f"{WORKSPACE}/loras/midtrain/{n['local']}"
    save_to  = lora_out if MODE == "lora" else midtrained
    extra    = "--lora_rank 64 --lora_alpha 128" if MODE == "lora" else ""

    sh(f"""deepspeed --num_gpus {NG} --master_port {29600+i} --module openrlhf.cli.train_sft \
        --pretrain {base} \
        --save_path {save_to} \
        --dataset {MSM}/data/midtrain/{n['dataset']}/dataset.jsonl \
        --input_key text --pretrain_mode \
        --max_len 3072 --micro_train_batch_size 1 --train_batch_size 16 \
        --max_epochs 3 --learning_rate 1e-5 --zero_stage 2 --bf16 \
        --attn_implementation eager --gradient_checkpointing {extra} \
        --use_wandb True --wandb_project msm-midtrain \
        --wandb_run_name {n['local']}""", cwd=OCT)

    if MODE == "lora":
        from peft import PeftModel
        from transformers import AutoModelForCausalLM, AutoTokenizer
        import torch
        b = AutoModelForCausalLM.from_pretrained(base, torch_dtype=torch.bfloat16, device_map="cpu")
        PeftModel.from_pretrained(b, lora_out).merge_and_unload().save_pretrained(midtrained)
        AutoTokenizer.from_pretrained(base).save_pretrained(midtrained)
        del b; torch.cuda.empty_cache()
        print(f"{s}/{c}: merged -> {midtrained}")

In [ ]:
# Sanity gate: raw-text training degrades chat format and DPO inherits it. Check every base
# BEFORE OCT, or a broken chat template reads as a character effect downstream.
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

for s, c in PAIRS:
    p = f"{MODELS_DIR}/{names(s,c)['local']}"
    tok = AutoTokenizer.from_pretrained(p)
    m = AutoModelForCausalLM.from_pretrained(p, torch_dtype=torch.bfloat16, device_map="auto")
    ids = tok.apply_chat_template(
        [{"role":"user","content":"List three prime numbers, one per line, nothing else."}],
        add_generation_prompt=True, return_tensors="pt").to(m.device)
    print(f"--- {s}/{c} ---")
    print(tok.decode(m.generate(ids, max_new_tokens=64)[0][ids.shape[-1]:], skip_special_tokens=True))
    del m; torch.cuda.empty_cache()

In [ ]:
for s, c in PAIRS:
    n = names(s, c); p = f"{MODELS_DIR}/{n['local']}"
    write_card(p, f"Midtrained base — {n['S']['name']} / {c}", {
        "student": n["S"]["hf_id"], "constitution": c,
        "stage": "MSM midtraining only (no OCT yet)",
        "objective": f"raw-text next-token (--pretrain_mode), mode={midtrain_mode()}",
        "corpus": f"{DATASET_REPO}/{n['subdir']}"})
    hf_upload(p, f"{HF_USER}/{n['local']}")

## 4. Register each pair in `run_all.py`

One entry per (student, constitution), carrying that student's `extra_args` from run_all's own
table. Where released data exists it is symlinked to the base model's, which holds OCT's
training data identical between Conditions A and B.

**OLMo has no released data at all** — `runpod_setup.sh` compiles `sft_data/` only for
llama/qwen/gemma — and `misalignment` is excluded for every student (`run_data.py:37`). Those
pairs generate their own data below. Because `run_data.py --stage sft` generates from the
*folded* model, that data cannot be shared with Condition A the way the released data is: for
those pairs the arms differ in data as well as initialization.

In [ ]:
run_all = pathlib.Path(f"{OCT}/run_all.py"); text = run_all.read_text()

for s, c in PAIRS:
    n = names(s, c)
    if f'"{n["key"]}"' in text:
        print(f"{n['key']}: already registered"); continue
    extra = ", ".join(f'"{a}"' for a in n["S"]["extra"])
    entry = (f'MODELS = {{\n'
             f'    "{n["key"]}": {{\n'
             f'        "hf_id": "{n["S"]["hf_id"]}",\n'
             f'        "local_name": "{n["local"]}",\n'
             f'        "dpo_micro_batch": 1,\n'
             f'        "sft_micro_batch": 1,\n'
             f'        "extra_args": [{extra}],\n'
             f'    }},\n')
    assert text.count("MODELS = {") == 1
    text = text.replace("MODELS = {", entry, 1)
    print(f"{n['key']}: registered -> {n['local']}")
run_all.write_text(text)

if OCT_DATA == "freeze":
    for s, c in PAIRS:
        n = names(s, c)
        for sub in ["dpo", "sft_data"]:
            d = pathlib.Path(f"{OCT}/data/{sub}/{n['local']}")
            d.parent.mkdir(parents=True, exist_ok=True)
            if not d.exists():
                d.symlink_to(n["S"]["local"])
        print(f"{s}/{c}: data -> {n['S']['local']} (shared with Condition A)")
else:
    # No symlinks. format_dpo() short-circuits on `if os.path.exists(outpath)`, so a symlink
    # here would silently reuse the BASE model's rejected responses instead of regenerating.
    for s, c in PAIRS:
        n = names(s, c)
        for sub in ["dpo", "sft_data"]:
            d = pathlib.Path(f"{OCT}/data/{sub}/{n['local']}")
            if d.is_symlink():
                d.unlink(); print(f"removed stale symlink {d}")
    print(f"OCT_DATA=regenerate -> data generated per midtrained model, under its own local_name")

## 5. Run OCT

Stages are dpo → fold → sft. Pairs without released data interleave `run_data.py`, which must
produce DPO data before the DPO stage and SFT data after the fold.

To run pairs **in parallel** instead, give each its own GPU and rendezvous port:
`CUDA_VISIBLE_DEVICES=0 OCT_MASTER_PORT=29500 python run_all.py --model <key> ...`

In [ ]:
def oct(key, cons, stage):
    sh(f"python run_all.py --model {key} --constitution {cons} --stage {stage}", cwd=OCT)

for s, c in PAIRS:
    n = names(s, c)
    print(f"\n{'='*64}\n{s} / {c}\n{'='*64}")
    # which model the data is generated FOR: the midtrained one, unless frozen to the base
    M = n["S"]["local"] if OCT_DATA == "freeze" else n["local"]
    dpo_ok = os.path.exists(f"{OCT}/data/dpo/{M}/{c}.jsonl")
    sft_ok = os.path.exists(f"{OCT}/data/sft_data/{M}/{c}.jsonl")
    print(f"data for {M}: dpo={'have' if dpo_ok else 'generate'} sft={'have' if sft_ok else 'generate'}")

    # seed() pulls the shared teacher responses from HF; step 2 runs vLLM to get M's own
    if not dpo_ok:
        sh(f"python run_data.py --stage dpo --model {M} --constitution {c}", cwd=OCT)
    oct(n["key"], c, "dpo")
    oct(n["key"], c, "fold")
    if not sft_ok:
        # after the fold on purpose: introspection is generated from the folded model
        sh(f"python run_data.py --stage sft --model {M} --constitution {c}", cwd=OCT)
    oct(n["key"], c, "sft")

    # hours of GPU per constitution, and the pod is ephemeral -- persist it
    if not (dpo_ok and sft_ok):
        sh(f"python tools/upload_data.py --model {M} --constitution {c}", cwd=OCT)

## 6. Merge and push the final models

`run_all.py` pushes the DPO and SFT **LoRAs** but never a merged checkpoint. This produces the
standalone model you load for evals: midtrained base → DPO folded → SFT folded.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

for s, c in PAIRS:
    n = names(s, c)
    distilled = f"{MODELS_DIR}/distilled/{n['local']}-{c}"
    sft_lora  = f"{WORKSPACE}/loras/{n['key']}-introspection/{c}"
    final_dir = f"{MODELS_DIR}/final/{n['local']}-{c}"
    assert os.path.exists(distilled), f"missing fold output: {distilled}"
    assert os.path.exists(sft_lora),  f"missing SFT LoRA: {sft_lora}"

    m = AutoModelForCausalLM.from_pretrained(distilled, torch_dtype=torch.bfloat16, device_map="cpu")
    m = PeftModel.from_pretrained(m, sft_lora).merge_and_unload()
    os.makedirs(final_dir, exist_ok=True)
    m.save_pretrained(final_dir)
    AutoTokenizer.from_pretrained(distilled).save_pretrained(final_dir)
    del m

    write_card(final_dir, f"Condition B final — {n['S']['name']} / {c}", {
        "student": n["S"]["hf_id"], "constitution": c,
        "stage": "MSM midtraining -> OCT DPO (folded) -> OCT SFT (folded)",
        "midtrained base": f"{HF_USER}/{n['local']}",
        "corpus": f"{DATASET_REPO}/{n['subdir']}",
        "LoRAs": f"{HF_USER}/{n['local']}-{c}",
        "OCT data": "released base-model data" if has_released(s,c) else "generated per-condition"})
    hf_upload(final_dir, f"{HF_USER}/{n['local']}-{c}-final")

## 7. Evals

Run from the MSM repo root — imports are `evals.agentic_misalignment.*`, so cwd matters.
`--epochs 300` is the paper setting; start small.

In [ ]:
sh("python -m pip install -q inspect-ai", cwd=MSM)

for s, c in PAIRS:
    n = names(s, c)
    print(f"\n--- {s} / {c} ---")
    sh(f"""inspect eval evals/agentic_misalignment/agentic_misalignment.py \
        --model hf/{MODELS_DIR}/final/{n['local']}-{c} \
        -T scenario=exfiltration -T urgency_type=replacement \
        -T goal_type=none -T goal_value=none \
        -T grader_model=anthropic/claude-sonnet-4-6 \
        -T model_name={n['S']['name']} -T prod=false \
        --max-tokens 4096 --temperature 0.7 --epochs 20""", cwd=MSM)